# Практика: Word2Vec и простой retrieval

В этом ноутбуке 5 задач:
1. Берём готовые эмбеддинги Word2Vec/аналог (через `gensim`) и смотрим ближайших соседей для слов из корпуса **20 Newsgroups**.
2. Обучаем свой **Skip-gram Word2Vec** на подкорпусе про **хоккей** и смотрим ближайшие слова к названиям команд.
3. Строим **индекс документов**: представляем документ как среднее Word2Vec (опционально с TF–IDF весами) и ищем ближайший документ по запросу.

> Подсказка: если что-то не скачивается  (нет интернета), используйте вариант обучения своих эмбеддингов на корпусе из пункта 2 и применяйте их в пунктах 1/3/4.


## 0) Установка и импорты

Если `gensim` или `faiss` не установлены в вашей среде, раскомментируйте `pip install` ниже.

In [ ]:
!pip -q install gensim
!pip -q install faiss-cpu  # для CPU-индекса (Windows иногда требует перезапуск kernel)

import re
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec


## 1) Данные: 20 Newsgroups

Загрузим корпус и подготовим простую токенизацию.

In [ ]:
newsgroups = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
texts = newsgroups.data
target_names = newsgroups.target_names

print("Документов:", len(texts))
print("Категорий:", len(target_names))
print("Пример категории:", target_names[0])

def tokenize(text):
    # Очень простая токенизация для учебных целей
    # 1) в нижний регистр
    # 2) оставляем только латиницу и цифры
    # 3) режем по пробелам
    text = text.lower()
    tokens = re.findall(r"[a-z0-9']+", text)
    return tokens

tokenized_texts = [tokenize(t) for t in texts]
print("Пример токенов:", tokenized_texts[0][:30])


Документов: 11314
Категорий: 20
Пример категории: alt.atheism
Пример токенов: ['i', 'was', 'wondering', 'if', 'anyone', 'out', 'there', 'could', 'enlighten', 'me', 'on', 'this', 'car', 'i', 'saw', 'the', 'other', 'day', 'it', 'was', 'a', '2', 'door', 'sports', 'car', 'looked', 'to', 'be', 'from', 'the']


## 2) Задание 1: готовые эмбеддинги и ближайшие соседи

Попробуем загрузить готовые вектора через `gensim.downloader`.

Рекомендуемые варианты:
- `glove-wiki-gigaword-100` (обычно быстрее скачивается)
- `word2vec-google-news-300` (очень большой)

Если скачивание недоступно — пропустите к заданию 2 (обучение своих эмбеддингов) и используйте их вместо готовых.

In [ ]:
#Посмотреть, какие модели доступны:
print(list(api.info()['models'].keys())[:30])

MODEL_NAME = "glove-wiki-gigaword-100"  # можно поменять

try:
    wv = api.load(MODEL_NAME)  # KeyedVectors
    print("Загружено:", MODEL_NAME)
    print("Размерность:", wv.vector_size)
except Exception as e:
    wv = None
    print("Не удалось загрузить готовые эмбеддинги:", repr(e))
    print("Продолжайте с обучением собственных эмбеддингов ниже (Задание 2).")


['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']
Загружено: glove-wiki-gigaword-100
Размерность: 100


### 2.1 Выбираем слова из корпуса и смотрим соседей

Выберите 10–20 слов (желательно тематических) и посмотрите ближайшие соседи.

**Задача:**
- Выберите по 3–5 слов из разных доменов (например, религия/спорт/компьютеры/медицина).
- Для каждого слова распечатайте 10 ближайших соседей.
- Сделайте 3–5 наблюдений: где соседи хорошие, где странные, почему так могло получиться.

### 2.2 (Доп.) fastText: готовые subword-эмбеддинги

Попробуем загрузить `fasttext-wiki-news-subwords-300` через `gensim.downloader`.
Это **subword fastText**: можно получать вектор для OOV-слов (в разумных пределах).
Если скачивание недоступно — пропустите эту часть (модель занимает 958 MB!).

In [ ]:
FT_MODEL_NAME = "fasttext-wiki-news-subwords-300"
try:
    ft_wv = api.load(FT_MODEL_NAME)  # KeyedVectors с subword поддержкой
    print("Загружено:", FT_MODEL_NAME)
    print("Размерность:", ft_wv.vector_size)

    # Пример: OOV слово (выдуманное/редкое). fastText обычно всё равно вернёт вектор
    oov_word = "skateboarders"  # попробуйте поменять на что-то ещё
    v = ft_wv.get_vector(oov_word)
    print("OOV vector ok, norm=", float(np.linalg.norm(v)))

    # Соседи для слова из словаря
    print("neighbors(hockey):", [w for w,_ in ft_wv.most_similar('hockey', topn=10)])
except Exception as e:
    ft_wv = None
    print("Не удалось загрузить fastText subwords:", repr(e))


Загружено: fasttext-wiki-news-subwords-300
Размерность: 300
OOV vector ok, norm= 0.694720983505249
neighbors(hockey): ['ice-hockey', 'icehockey', 'hockey-related', 'hockey-playing', 'hocky', 'hockey-mad', 'field-hockey', 'hocket', 'non-hockey', 'Hockey']


### 2.3 Семантические сдвиги (vector offsets)

Идея: отношения между словами часто кодируются **разностями векторов**.

**Что проверить:**
- Сдвиг `queen - king` похож на `woman - man`
- Сдвиг `paris - france` похож на `berlin - germany` (и т.п.)

Ниже мы:
1) найдём ближайшие слова к вектору-сдвигу (например, к `queen-king`),
2) посчитаем косинусное сходство между двумя сдвигами.


In [ ]:
def pick_embeddings_for_offsets():
    # Приоритет: готовые wv -> fastText ft_wv -> None
    if 'wv' in globals() and wv is not None:
        return wv, 'pretrained (wv)'
    if 'ft_wv' in globals() and ft_wv is not None:
        return ft_wv, 'fastText subwords (ft_wv)'
    return None, 'none'

emb, emb_name = pick_embeddings_for_offsets()
print('Embeddings for offsets:', emb_name)

def unit(v):
    v = v.astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-9)

def nearest_to_vector(emb, vec, topn=10, ban=()):
    # gensim KeyedVectors умеют most_similar по вектору
    res = emb.most_similar([vec], topn=topn + len(ban))
    out = []
    for w, s in res:
        if w not in ban:
            out.append((w, s))
        if len(out) >= topn:
            break
    return out

def offset(word_a, word_b):
    # vec(word_a) - vec(word_b)
    return emb.get_vector(word_a) - emb.get_vector(word_b)

def try_offset_demo(pair1, pair2, topn=10):
    (a1, b1) = pair1
    (a2, b2) = pair2
    missing = [w for w in [a1,b1,a2,b2] if w not in emb]
    if missing:
        print('Пропускаю, нет слов в словаре:', missing)
        return
    d1 = offset(a1, b1)
    d2 = offset(a2, b2)
    sim = float(unit(d1) @ unit(d2))
    print(f"offset({a1}-{b1}) vs offset({a2}-{b2}) cosine = {sim:.3f}")
    print(f"Nearest to vector ({a1}-{b1}):", [w for w,_ in nearest_to_vector(emb, d1, topn=topn, ban=(a1,b1))])
    print(f"Nearest to vector ({a2}-{b2}):", [w for w,_ in nearest_to_vector(emb, d2, topn=topn, ban=(a2,b2))])
    print()

if emb is None:
    print('Нет подходящих эмбеддингов. Сначала загрузите wv или ft_wv.')
else:
    # 1) queen-king ~ woman-man
    try_offset_demo(('queen','king'), ('woman','man'), topn=10)

    # 2) paris-france ~ berlin-germany (и ещё несколько примеров)
    try_offset_demo(('paris','france'), ('berlin','germany'), topn=10)
    try_offset_demo(('rome','italy'), ('madrid','spain'), topn=10)

    # 3) Классическая аналогия: king - man + woman ≈ queen
    needed = ['king','man','woman']
    if all(w in emb for w in needed):
        v = emb.get_vector('king') - emb.get_vector('man') + emb.get_vector('woman')
        print("Analogy king - man + woman ->", nearest_to_vector(emb, v, topn=10, ban=('king','man','woman'))[:10])
    else:
        print('Пропускаю analogy king-man+woman: не хватает слов')


Embeddings for offsets: pretrained (wv)
offset(queen-king) vs offset(woman-man) cosine = 0.451
Nearest to vector (queen-king): ['coppered', 'dessay', 'actress/singer', 'voluptuous', 'ladies-in-waiting', 'jointure', 'tyresö', 'daylesford', 'mildura', 'blonde']
Nearest to vector (woman-man): ['menstruating', 'miscarry', 'lactating', 'pre-eclampsia', 'uterus', 'ovulating', 'miscarried', 'adoptee', 'comnena', 'pulecio']

offset(paris-france) vs offset(berlin-germany) cosine = 0.782
Nearest to vector (paris-france): ['ryōgoku', 'heliopolis', 'midtown', 'uptown', 'skavsta', 'soho', 'haymarket', 'shiodome', 'kuangdi', 'brockport']
Nearest to vector (berlin-germany): ['haymarket', 'inforadio', 'volksbühne', 'frightfest', 'odéon', 'adelphi', 'terazije', 'vgik', 'bobino', 'comicon']

offset(rome-italy) vs offset(madrid-spain) cosine = 0.523
Nearest to vector (rome-italy): ['inova', 'canons', 'lambeth', 'malacanang', 'sobchak', 'shattuck', '.2009', 'rosenblatt', 'standiford', 'prebend']
Nearest t

In [ ]:
def show_neighbors(wv, word, topn=10):
    if wv is None:
        print("Нет загруженных эмбеддингов (wv=None).")
        return
    if word not in wv:
        print(f"'{word}' нет в словаре модели.")
        return
    neigh = wv.most_similar(word, topn=topn)
    print(word, "->", [w for w, _ in neigh])

words_to_try = ["hockey", "windows", "jesus", "doctor", "encryption", "space", "graphics"]

for w in words_to_try:
    show_neighbors(wv, w, topn=10)


hockey -> ['basketball', 'football', 'nhl', 'soccer', 'baseball', 'league', 'skating', 'lacrosse', 'team', 'games']
windows -> ['window', 'xp', 'desktop', 'doors', 'browser', 'computers', 'microsoft', 'roof', 'walls', 'server']
jesus -> ['christ', 'god', 'resurrection', 'crucifixion', 'crucified', 'faith', 'holy', 'divine', 'bible', 'disciples']
doctor -> ['physician', 'nurse', 'dr.', 'doctors', 'patient', 'medical', 'surgeon', 'hospital', 'psychiatrist', 'dentist']
encryption -> ['cryptographic', 'authentication', 'encrypted', 'decryption', 'cryptography', 'firewalls', 'public-key', 'firewall', 'proprietary', 'algorithms']
space -> ['nasa', 'spaces', 'shuttle', 'earth', 'spacecraft', 'orbit', 'module', 'astronauts', 'spaceship', 'center']
graphics -> ['layouts', 'graphic', '3d', 'multimedia', 'graphical', 'animation', 'animations', 'hardware', 'editing', 'software']


## 3) Задание 2: обучаем Word2Vec на хоккейной категории

Возьмём подкорпус `rec.sport.hockey` и обучим **Skip-gram**.

**Задача:**
- Обучите модель.
- Проверьте ближайшие слова к названиям команд: `rangers`, `bruins`, `canadiens`, `flyers`, `leafs`, `oilers` и т.п.
- Сравните с готовыми эмбеддингами (если они доступны).

In [ ]:
# Вытащим только хоккей
hockey = fetch_20newsgroups(subset="train",
                            categories=["rec.sport.hockey"],
                            remove=("headers", "footers", "quotes"))
hockey_texts = hockey.data
hockey_tok = [tokenize(t) for t in hockey_texts]

print("Хоккейных документов:", len(hockey_tok))
print("Пример токенов:", hockey_tok[0][:30])


Хоккейных документов: 600
Пример токенов: ['it', 'looks', 'like', 'the', 'edmonton', 'oilers', 'just', 'decided', 'to', 'take', 'a', 'european', 'vacation', 'this', 'spring', 'ranford', 'tugnutt', 'benning', 'manson', 'smith', 'buchberger', 'and', 'corson', 'are', 'playing', 'for', 'canada', 'podein', 'and', 'weight']


In [ ]:
# Обучение skip-gram: sg=1
# Параметры можно менять
w2v_hockey = Word2Vec(
    sentences=hockey_tok,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=10,
    epochs=10,
    workers=4
)

wv_hockey = w2v_hockey.wv
print("Словарь хоккейной модели:", len(wv_hockey))
print("Размерность:", wv_hockey.vector_size)


Словарь хоккейной модели: 4247
Размерность: 100


In [ ]:
teams = ["rangers", "bruins", "canadiens", "flyers", "leafs", "oilers", "penguins", "flames", "canucks"]
for t in teams:
    if t in wv_hockey:
        print("\n", t, "->", [w for w, _ in wv_hockey.most_similar(t, topn=10)])
    else:
        print("\n", t, "нет в словаре (проверь min_count или орфографию)")



 rangers -> ['capitals', 'keller', "let's", 'kovalev', 'york', 'zubov', 'keith', 'ny', 'quakers', 'umaine']

 bruins -> ['habs', 'cd', 'indians', 'umaine', 'boston', 'providence', 'springfield', 'hershey', 'hammerl', 'southern']

 canadiens -> ['cleveland', 'atlanta', 'northern', 'hl', 'nordiques', 'col', 'atlantic', 'cincinnati', 'hamilton', 'southern']

 flyers -> ['kept', 'killing', 'pressure', 'smothered', 'swedes', 'sabres', 'kill', 'straight', 'advantage', 'lssu']

 leafs -> ['maple', 'potvin', 'toronto', 'problems', 'wings', "detroit's", "john's", 'smothered', 'hawks', 'powerplay']

 oilers -> ['breton', 'cape', 'saint', 'edmonton', 'citadels', 'fredericton', 'skipjacks', 'district', 'halifax', 'hershey']

 penguins -> ['considerations', 'pittsburgh', 'pitiful', 'future', 'lineup', 'pa', 'whalers', 'nordiques', 'publicly', 'trade']

 flames -> ['vernon', '1980', 'calgary', 'kings', 'nilsson', 'convert', 'hrudey', 'saint', 'exhibition', 'blue']

 canucks -> ['hamilton', 'blackha

## 4) Задание 3: индекс документов как матрица (mean Word2Vec)

Построим эмбеддинг документа как **среднее** эмбеддингов слов.

Два варианта:
1. Просто среднее.
2. **TF–IDF взвешенное** среднее (обычно лучше).

**Задача:**
- Постройте матрицу `D` размера (num_docs × dim).
- Напишите 3–5 запросов и посмотрите топ-5 ближайших документов.
- Оцените глазами: насколько выдача “в тему”.

In [ ]:
# Для retrieval возьмём небольшой поднабор, чтобы было быстрее
X_train, X_test = train_test_split(texts, test_size=0.2, random_state=42)
docs = X_test[:1000]  # ограничим

docs_tok = [tokenize(t) for t in docs]

# Выберем, какие word vectors использовать:
# 1) если есть готовые (wv) — можно их;
# 2) иначе используем хоккейные (wv_hockey) или обучите Word2Vec на всем корпусе.
if wv is not None:
    wv_use = wv
    print("Используем готовые эмбеддинги:", MODEL_NAME)
else:
    wv_use = wv_hockey
    print("Используем эмбеддинги, обученные на хоккейном подкорпусе (wv_hockey).")

DIM = wv_use.vector_size

def doc_vector_mean(tokens, wv_use):
    vecs = [wv_use[w] for w in tokens if w in wv_use]
    if not vecs:
        return np.zeros(DIM, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# TF–IDF для взвешивания
vectorizer = TfidfVectorizer(tokenizer=tokenize, lowercase=True, min_df=2, max_df=0.9)
tfidf = vectorizer.fit_transform(docs)  # (num_docs, vocab)
vocab = vectorizer.vocabulary_  # word -> col

def doc_vector_tfidf(tokens, wv_use, tfidf_row, vocab):
    # tfidf_row: sparse row (1, vocab_size)
    weights = {}
    # берём только слова из tokens, которые есть в vocab
    for w in tokens:
        j = vocab.get(w, None)
        if j is not None:
            weights[w] = tfidf_row[0, j]
    # соберём взвешенную сумму
    num = np.zeros(DIM, dtype=np.float32)
    den = 0.0
    for w, a in weights.items():
        if w in wv_use and a > 0:
            num += (a * wv_use[w]).astype(np.float32)
            den += float(a)
    if den == 0.0:
        return doc_vector_mean(tokens, wv_use)
    return (num / den).astype(np.float32)

# Матрица эмбеддингов документов
D_mean = np.vstack([doc_vector_mean(t, wv_use) for t in docs_tok])
D_tfidf = np.vstack([doc_vector_tfidf(docs_tok[i], wv_use, tfidf[i], vocab) for i in range(len(docs_tok))])

print("D_mean shape:", D_mean.shape, "D_tfidf shape:", D_tfidf.shape)


# Нормируем матрицы заранее для cosine (ускоряет поиск)
D_mean_n = D_mean / (np.linalg.norm(D_mean, axis=1, keepdims=True) + 1e-9)
D_tfidf_n = D_tfidf / (np.linalg.norm(D_tfidf, axis=1, keepdims=True) + 1e-9)
print('D_mean_n shape:', D_mean_n.shape, 'D_tfidf_n shape:', D_tfidf_n.shape)


Используем готовые эмбеддинги: glove-wiki-gigaword-100


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


D_mean shape: (1000, 100) D_tfidf shape: (1000, 100)
D_mean_n shape: (1000, 100) D_tfidf_n shape: (1000, 100)


In [ ]:
def normalize_rows(M: np.ndarray) -> np.ndarray:
    M = M.astype(np.float32, copy=False)
    norms = np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M / norms

def cosine_topk_pre_norm(query_vec: np.ndarray, Dnrm: np.ndarray, k: int = 5):
    """Cosine top-k, where Dnrm already has unit-norm rows."""
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = Dnrm @ q
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]


In [ ]:
# 5 запросов + вызовы: TF–IDF baseline + dense cosine

import re
import numpy as np

queries = [
    "windows driver installation error",
    "hockey playoffs game score team",
    "space shuttle mission nasa orbit",
    "computer graphics rendering polygon",
    "medical doctor treatment symptoms",
]
K = 5

for q in queries:
    print("=" * 90)
    print("Query:", q)

    # (1) TF–IDF baseline: scores = tfidf @ q_tfidf^T
    q_tfidf = vectorizer.transform([q])                    # (1, |V|)
    scores_tfidf = (tfidf @ q_tfidf.T).toarray().ravel()   # (N,)
    idx_tfidf = np.argsort(-scores_tfidf)[:K]

    print("\nTF–IDF top-k:")
    for r, i in enumerate(idx_tfidf, 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={scores_tfidf[i]:.4f} | {snippet}...")

    # (2) Dense cosine over pre-normalized doc matrix D_tfidf_n
    tok = tokenize(q)
    qv = doc_vector_tfidf(tok, wv_use, q_tfidf, vocab)     # (dim,)
    idx_dense, sc_dense = cosine_topk_pre_norm(qv, D_tfidf_n, k=K)

    print("\nDense (cosine) top-k:")
    for r, (i, s) in enumerate(zip(idx_dense, sc_dense), 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={float(s):.4f} | {snippet}...")

Query: windows driver installation error

TF–IDF top-k:
 1. score=0.3135 | Hi! I need a Windows 3.1 driver for the Matrox PG-1281 CV SVGA card. At the moment Windows runs only in the 640x480 mode. If you have a driver for this card, please send it with the OEMSETUP.INF to bo...
 2. score=0.2505 |  How are you attempting to do that? Are you using the DIS_PKT9 program? This provides a packet driver on top of the NDIS driver. ...
 3. score=0.2355 | I quit windows normally to run a special DOS app, got done with it and tried to start windows. Ok got the title screen, Windows background, DOS with an error about loading PROGMAN.EXE. Hum, yep PROGMA...
 4. score=0.2050 | stuff deleted stuff deleted The problem mentioned last is a known Quadra SCSI problem, it was heavily discussed last year and an Apple employee pointed out that there was a one byte error in the SCSI ...
 5. score=0.1868 | Hello, I'm trying to get X11R5 running on my PC and ran into the following error message when trying to 

## 5) Задание 4: тот же retrieval, но через FAISS

FAISS делает быстрый поиск ближайших соседей.

Мы будем использовать cosine similarity как inner product после нормировки векторов.

In [ ]:
try:
    import faiss
    faiss_available = True
    print("FAISS импортирован:", faiss.__version__ if hasattr(faiss, "__version__") else "ok")
except Exception as e:
    faiss_available = False
    print("FAISS не доступен:", repr(e))
    print("Если можно, установите: pip install faiss-cpu")


FAISS импортирован: 1.13.2


In [ ]:
if faiss_available:
    # Нормируем документные вектора (cosine -> inner product)
    D = D_tfidf_n.astype(np.float32)  # уже нормировано

    index = faiss.IndexFlatIP(D.shape[1])
    index.add(D)
    print("В индексе документов:", index.ntotal)

    def search_faiss(query, k=5):
        tok = tokenize(query)
        q_tfidf = vectorizer.transform([query])
        qv = doc_vector_tfidf(tok, wv_use, q_tfidf, vocab).astype(np.float32)
        qv /= (np.linalg.norm(qv) + 1e-9)
        scores, idx = index.search(qv.reshape(1, -1), k)
        idx = idx[0]
        scores = scores[0]
        print("Query:", query)
        for rank, (i, s) in enumerate(zip(idx, scores), 1):
            snippet = re.sub(r"\s+", " ", docs[int(i)])[:200]
            print(f"{rank:>2}. score={float(s):.3f} | {snippet}...")
        print()

    for q in queries:
        search_faiss(q, k=5)


В индексе документов: 1000
Query: windows driver installation error
 1. score=0.781 |  I think George is referring to switch.zip in the ~ftp/pub/pc/win3/drivers/video directory. Description reads -- Switcher: Windows Video Mode Switcher. ...
 2. score=0.775 | Hi, I have a 486/66MHz SYS based PC with 8M RAM and a problem. What is the best way to configure high memory with QEMM/386MAX ?? I have a SPEEDSTAR 24X video card and use Hyperdisk disk cache software...
 3. score=0.768 | Auto Logic Panasonic answering machine with dual cassette system. I will include cassettes and AC power adaptor. Excellent condition. Asking $30 with accessories. ...
 4. score=0.767 | Hi! I need a Windows 3.1 driver for the Matrox PG-1281 CV SVGA card. At the moment Windows runs only in the 640x480 mode. If you have a driver for this card, please send it with the OEMSETUP.INF to bo...
 5. score=0.764 | I just got a Quadra 800 8/230 and I've noticed that I can't change the desktop color from the beautiful gray. I

### Задание
1) Загрузите модель эмбеддингов для русского языка и проанализируйте, какой препроцессинг требуется для успешной работы с ней.  
2) Подготовьте собственный датасет на русском языке, обучите **word2vec** и сравните ближайших соседей с готовыми эмбеддингами.  
3) Постройте индекс по вашему датасету. Если ваши данные — одно длинное произведение, разбейте его на части (абзацы или фрагменты по 100–500 слов). Проверьте работу поиска по вашему индексу на нескольких запросах.

### Что сдавать
1) Список выбранных слов и их ближайшие соседи (готовая русская модель; задание 1).  
2) 5–10 примеров ближайших слов + 2–3 наблюдения (word2vec на ваших данных; задание 2).  
3) Для retrieval: 3–5 запросов и топ-5 документов + короткий вывод (задание 3).
